In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import plotly.express as px
import random

In [2]:
data = pd.read_csv('Pokemon.csv')

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   #           800 non-null    int64 
 1   Name        800 non-null    object
 2   Type 1      800 non-null    object
 3   Type 2      414 non-null    object
 4   Total       800 non-null    int64 
 5   HP          800 non-null    int64 
 6   Attack      800 non-null    int64 
 7   Defense     800 non-null    int64 
 8   Sp. Atk     800 non-null    int64 
 9   Sp. Def     800 non-null    int64 
 10  Speed       800 non-null    int64 
 11  Generation  800 non-null    int64 
 12  Legendary   800 non-null    bool  
dtypes: bool(1), int64(9), object(3)
memory usage: 75.9+ KB


In [3]:
data['Power'] = data['Type 2'].fillna(data['Type 1'])

pre_processed_data = data.drop(columns=['Type 1', 'Type 2'], inplace=False)


In [ ]:

stat_columns = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
num_samples_per_pokemon = 100 

augmented_data = []

for index, row in pre_processed_data.iterrows():

    original_row = row.to_dict()
    original_row['Is_Augmented'] = 0 
    augmented_data.append(original_row)
    
    for _ in range(num_samples_per_pokemon - 1): 
        
        new_row = row.to_dict()
        
        for stat in stat_columns:

            base_val = new_row[stat]
            noise_val = np.random.normal(loc=base_val, scale=base_val * 0.05)
            new_row[stat] = max(1, int(round(noise_val)))

            
            
        new_row['Total'] = sum(new_row[stat] for stat in stat_columns)
        new_row['Is_Augmented'] = 1
        augmented_data.append(new_row)


augmented_df = pd.DataFrame(augmented_data)

augmented_df['Legendary'] = augmented_df['Legendary'].astype(int)

augmented_df.to_csv('augmented_pokemon.csv', index=False)

print(f"Original shape: {pre_processed_data.shape}")
print(f"Augmented shape: {augmented_df.shape}")

Original shape: (800, 12)
Augmented shape: (80000, 13)


In [5]:
name = random.choice(data["Name"].unique())
print(f"Randomly selected Pokémon: {name}")

print(data[data["Name"]==name])

augmented_df[augmented_df["Name"]==name].sample(20)

Randomly selected Pokémon: Machamp
     #     Name    Type 1 Type 2  Total  HP  Attack  Defense  Sp. Atk  \
74  68  Machamp  Fighting    NaN    505  90     130       80       65   

    Sp. Def  Speed  Generation  Legendary     Power  
74       85     55           1      False  Fighting  


,#,Name,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary,Power,Is_Augmented
7452,68,Machamp,496,80,132,81,67,81,55,1,0,Fighting,1
7497,68,Machamp,513,86,135,88,61,87,56,1,0,Fighting,1
7498,68,Machamp,511,88,130,86,65,86,56,1,0,Fighting,1
7458,68,Machamp,520,86,136,89,66,85,58,1,0,Fighting,1
7436,68,Machamp,510,85,137,77,64,91,56,1,0,Fighting,1
7471,68,Machamp,501,87,130,77,67,83,57,1,0,Fighting,1
7483,68,Machamp,502,90,130,77,68,78,59,1,0,Fighting,1
7447,68,Machamp,512,92,124,81,70,91,54,1,0,Fighting,1
7496,68,Machamp,506,88,134,78,67,83,56,1,0,Fighting,1
7456,68,Machamp,497,81,129,79,67,88,53,1,0,Fighting,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X1_without_power = augmented_df[['HP' , 'Attack' , 'Defense' , 'Sp. Atk' , 'Sp. Def' , 'Speed']]
X2_with_power = augmented_df[['HP' , 'Attack' , 'Defense' , 'Sp. Atk' , 'Sp. Def' , 'Speed' ,'Legendary','Generation', 'Power']]

X2_with_power = pd.get_dummies(X2_with_power, columns=['Power'] , drop_first=True)

y_power_label = augmented_df['Power']
y_name_label = augmented_df['Name']
y_legendary_label = augmented_df['Legendary']
y_generation_label = augmented_df['Generation']


power_encoder = LabelEncoder()
name_encoder = LabelEncoder()
legendary_encoder = LabelEncoder()
generation_encoder = LabelEncoder()


y_power_encoded = power_encoder.fit_transform(y_power_label)
y_name_encoded = name_encoder.fit_transform(y_name_label)
y_legendary_encoded = legendary_encoder.fit_transform(y_legendary_label)
y_generation_encoded = generation_encoder.fit_transform(y_generation_label)



# X_train_topower  , X_test_topower, y_train_power, y_test_power = train_test_split(X1_without_power, y_power_encoded, test_size=0.1, random_state=42 , stratify=y_power_encoded , shuffle=True)

# X_train_toname, X_test_toname, y_train_name, y_test_name = train_test_split(X2_with_power, y_name_encoded, test_size=0.1, random_state=42 , stratify=y_name_encoded , shuffle=True)

# X_train_tolegendary, X_test_tolegendary, y_train_legendary, y_test_legendary = train_test_split(X1_without_power, y_legendary_encoded, test_size=0.1, random_state=42 , stratify=y_legendary_encoded , shuffle=True)

# X_train_togeneration, X_test_togeneration, y_train_generation, y_test_generation = train_test_split(X1_without_power, y_generation_encoded, test_size=0.1, random_state=42 , stratify=y_generation_encoded , shuffle=True)



In [7]:
training_columns = X2_with_power.columns.tolist()
print(f"Model 2 expects {len(training_columns)} columns:")
print(training_columns)

Model 2 expects 25 columns:
['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Legendary', 'Generation', 'Power_Dark', 'Power_Dragon', 'Power_Electric', 'Power_Fairy', 'Power_Fighting', 'Power_Fire', 'Power_Flying', 'Power_Ghost', 'Power_Grass', 'Power_Ground', 'Power_Ice', 'Power_Normal', 'Power_Poison', 'Power_Psychic', 'Power_Rock', 'Power_Steel', 'Power_Water']


In [8]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score



# model_power_predictor = RandomForestClassifier()

# model_power_predictor.fit(X1_without_power, y_train_power)
# y_pred_power = model_power_predictor.predict(X_test_topower)


# accuracy = accuracy_score(y_test_power, y_pred_power)
# precision = precision_score(y_test_power, y_pred_power, average='weighted')
# recall = recall_score(y_test_power, y_pred_power, average='weighted')
# f1 = f1_score(y_test_power, y_pred_power, average='weighted')


# print("Model 1 (Predicting Power):")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1 Score: {f1:.4f}")

# print("=" * 30)


# model_legendary_predictor = RandomForestClassifier()

# model_legendary_predictor.fit(X_train_tolegendary, y_train_legendary)
# y_pred_legendary = model_legendary_predictor.predict(X_test_tolegendary)


# accuracy = accuracy_score(y_test_legendary, y_pred_legendary)
# precision = precision_score(y_test_legendary, y_pred_legendary, average='weighted')
# recall = recall_score(y_test_legendary, y_pred_legendary, average='weighted')
# f1 = f1_score(y_test_legendary, y_pred_legendary, average='weighted')


# print("Model 2 (Predicting Legendary):")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1 Score: {f1:.4f}")

# print("=" * 30)

# model_generation_predictor = RandomForestClassifier()

# model_generation_predictor.fit(X_train_togeneration, y_train_generation)
# y_pred_generation = model_generation_predictor.predict(X_test_togeneration)


# accuracy = accuracy_score(y_test_generation, y_pred_generation)
# precision = precision_score(y_test_generation, y_pred_generation, average='weighted')
# recall = recall_score(y_test_generation, y_pred_generation, average='weighted')
# f1 = f1_score(y_test_generation, y_pred_generation, average='weighted')


# print("Model 3 (Predicting Generation):")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1 Score: {f1:.4f}")

# print("=" * 30)

# model_name_predictor = RandomForestClassifier()

# model_name_predictor.fit(X_train_toname, y_train_name)
# y_pred_name = model_name_predictor.predict(X_test_toname)


# accuracy2 = accuracy_score(y_test_name, y_pred_name)
# precision2 = precision_score(y_test_name, y_pred_name, average='weighted')
# recall2 = recall_score(y_test_name, y_pred_name, average='weighted')
# f1_2 = f1_score(y_test_name, y_pred_name, average='weighted')

# print("Model 4 (Predicting Name):")
# print(f"Accuracy: {accuracy2:.4f}")
# print(f"Precision: {precision2:.4f}")
# print(f"Recall: {recall2:.4f}")
# print(f"F1 Score: {f1_2:.4f}")



Model 1 (Predicting Power):
- Accuracy: 0.9539
- Precision: 0.9539
- Recall: 0.9539
- F1 Score: 0.9538

==============================

Model 2 (Predicting Legendary):
- Accuracy: 0.9915
- Precision: 0.9914
- Recall: 0.9915
- F1 Score: 0.9915

==============================

Model 3 (Predicting Generation):
- Accuracy: 0.9599
- Precision: 0.9599
- Recall: 0.9599
- F1 Score: 0.9599

==============================

Model 4 (Predicting Name):
- Accuracy: 0.9935
- Precision: 0.9935
- Recall: 0.9935
- F1 Score: 0.9934

In [9]:
from sklearn.ensemble import RandomForestClassifier


model_power_predictor = RandomForestClassifier()
model_power_predictor.fit(X1_without_power, y_power_encoded)


model_legendary_predictor = RandomForestClassifier()
model_legendary_predictor.fit(X1_without_power, y_legendary_encoded)


model_generation_predictor = RandomForestClassifier()
model_generation_predictor.fit(X1_without_power, y_generation_encoded)


model_name_predictor = RandomForestClassifier()
model_name_predictor.fit(X2_with_power, y_name_encoded)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
import joblib

filename1 = 'PowerPredictor.pkl'
filename2 = 'NamePredictor.pkl'
filename3 = 'LegendaryPredictor.pkl'
filename4 = 'GenerationPredictor.pkl'

joblib.dump(model_power_predictor, filename1 , compress=3)
print(f"Model saved as {filename1}")
joblib.dump(model_name_predictor, filename2 ,  compress=3)
print(f"Model saved as {filename2}")
joblib.dump(model_legendary_predictor, filename3 ,  compress=3)
print(f"Model saved as {filename3}")
joblib.dump(model_generation_predictor, filename4 ,  compress=3)
print(f"Model saved as {filename4}")

filename1 = 'power_encoder.pkl'
filename2 = 'name_encoder.pkl'
filename3 = 'legendary_encoder.pkl'
filename4 = 'generation_encoder.pkl'


joblib.dump(power_encoder, filename1 , compress=3)
print(f"Model saved as {filename1}")
joblib.dump(name_encoder, filename2 ,  compress=3)
print(f"Model saved as {filename2}")
joblib.dump(legendary_encoder, filename3 ,  compress=3)
print(f"Model saved as {filename3}")
joblib.dump(generation_encoder, filename4 ,  compress=3)
print(f"Model saved as {filename4}")


Model saved as PowerPredictor.pkl
Model saved as NamePredictor.pkl
Model saved as LegendaryPredictor.pkl
Model saved as GenerationPredictor.pkl
Model saved as power_encoder.pkl
Model saved as name_encoder.pkl
Model saved as legendary_encoder.pkl
Model saved as generation_encoder.pkl


In [11]:
import requests
from IPython.display import Image, display, HTML

# Build name -> Pokédex ID map directly from your CSV
name_to_id = dict(zip(data['Name'], data['#']))

print(f"Lookup table ready: {len(name_to_id)} Pokémon")

def show_pokemon(predicted_name):
    pokedex_id = name_to_id.get(predicted_name)
    if pokedex_id is None:
        print(f"Not found: {predicted_name}")
        return
    
    url = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/{pokedex_id}.png"
    display(HTML(f'<img src="{url}" width="200px"/>'))

Lookup table ready: 800 Pokémon


In [12]:
PowerPredictor = joblib.load('PowerPredictor.pkl')
LegendaryPredictor = joblib.load('LegendaryPredictor.pkl')
GenerationPredictor = joblib.load('GenerationPredictor.pkl')
NamePredictor = joblib.load('NamePredictor.pkl')

power_encoder = joblib.load('power_encoder.pkl')
name_encoder = joblib.load('name_encoder.pkl')
legendary_encoder = joblib.load('legendary_encoder.pkl')
generation_encoder = joblib.load('generation_encoder.pkl')


In [13]:
health_point = random.randint(augmented_df['HP'].min(), augmented_df['HP'].max())
print(f"Randomly selected HP value: {health_point}")

attack = random.randint(augmented_df['Attack'].min(), augmented_df['Attack'].max())
print(f"Randomly selected Attack value: {attack}")

defense = random.randint(augmented_df['Defense'].min(), augmented_df['Defense'].max())
print(f"Randomly selected Defense value: {defense}")

sp_atk = random.randint(augmented_df['Sp. Atk'].min(), augmented_df['Sp. Atk'].max())
print(f"Randomly selected Sp. Atk value: {sp_atk}")

sp_def = random.randint(augmented_df['Sp. Def'].min(), augmented_df['Sp. Def'].max())
print(f"Randomly selected Sp. Def value: {sp_def}")

speed = random.randint(augmented_df['Speed'].min(), augmented_df['Speed'].max())
print(f"Randomly selected Speed value: {speed}")



new_pokemon = pd.DataFrame({
    'HP': [health_point],
    'Attack': [attack],
    'Defense': [defense],
    'Sp. Atk': [sp_atk],
    'Sp. Def': [sp_def],
    'Speed': [speed]
    })



# new_pokemon = pd.DataFrame({
#     'HP': [78],
#     'Attack': [84],
#     'Defense': [78],
#     'Sp. Atk': [100],
#     'Sp. Def': [85],
#     'Speed': [100],
#     })


predicted_power_encoded = PowerPredictor.predict(new_pokemon)[0]
predicted_power = power_encoder.inverse_transform([predicted_power_encoded])[0]

print(f"Predicted Power: {predicted_power}")

predicted_legendary_encoded = LegendaryPredictor.predict(new_pokemon)[0]
predicted_legendary = legendary_encoder.inverse_transform([predicted_legendary_encoded])[0]

predicted_generation_encoded = GenerationPredictor.predict(new_pokemon)[0]
predicted_generation = generation_encoder.inverse_transform([predicted_generation_encoded])[0]

new_pokemon['Legendary'] = predicted_legendary
new_pokemon['Generation'] = predicted_generation
new_pokemon[[f'Power_{predicted_power}']] = 1


new_pokemon = new_pokemon.reindex(columns=training_columns, fill_value=0)


predicted_name_encoded = NamePredictor.predict(new_pokemon)[0]
predicted_name = name_encoder.inverse_transform([predicted_name_encoded])[0]

print(f"Predicted Pokémon: {predicted_name}")
print(f"Pokemon Power is: {augmented_df[augmented_df['Name'] == predicted_name]['Power'].iloc[0]}")

show_pokemon(predicted_name)

new_pokemon[[f'Power_{predicted_power}']]

Randomly selected HP value: 54
Randomly selected Attack value: 130
Randomly selected Defense value: 102
Randomly selected Sp. Atk value: 10
Randomly selected Sp. Def value: 58
Randomly selected Speed value: 110
Predicted Power: Steel
Predicted Pokémon: Bonsly
Pokemon Power is: Rock


,Power_Steel
0,1


In [ ]:
from tqdm import tqdm

def PokemonPredictor(health_point = None , attack = None , defense = None , sp_atk = None , sp_def = None , speed = None) : 

    if health_point is None:
        health_point = random.randint(augmented_df['HP'].min(), augmented_df['HP'].max())

    if attack is None:
        attack = random.randint(augmented_df['Attack'].min(), augmented_df['Attack'].max())

    if defense is None:
        defense = random.randint(augmented_df['Defense'].min(), augmented_df['Defense'].max())

    if sp_atk is None:
        sp_atk = random.randint(augmented_df['Sp. Atk'].min(), augmented_df['Sp. Atk'].max())

    if sp_def is None:
        sp_def = random.randint(augmented_df['Sp. Def'].min(), augmented_df['Sp. Def'].max())

    if speed is None:
        speed = random.randint(augmented_df['Speed'].min(), augmented_df['Speed'].max())



    new_pokemon = pd.DataFrame({
        'HP': [health_point],
        'Attack': [attack],
        'Defense': [defense],
        'Sp. Atk': [sp_atk],
        'Sp. Def': [sp_def],
        'Speed': [speed]
        })
    

    predicted_power_encoded = PowerPredictor.predict(new_pokemon)[0]
    predicted_power = power_encoder.inverse_transform([predicted_power_encoded])[0]

    predicted_legendary_encoded = LegendaryPredictor.predict(new_pokemon)[0]
    predicted_legendary = legendary_encoder.inverse_transform([predicted_legendary_encoded])[0]

    predicted_generation_encoded = GenerationPredictor.predict(new_pokemon)[0]
    predicted_generation = generation_encoder.inverse_transform([predicted_generation_encoded])[0]

    new_pokemon['Legendary'] = predicted_legendary
    new_pokemon['Generation'] = predicted_generation
    new_pokemon[[f'Power_{predicted_power}']] = 1


    new_pokemon = new_pokemon.reindex(columns=training_columns, fill_value=0)


    predicted_name_encoded = NamePredictor.predict(new_pokemon)[0]
    predicted_name = name_encoder.inverse_transform([predicted_name_encoded])[0]

    
    return predicted_name 


team = []

for _ in tqdm(range(10000), desc="Generating team"):

    pokemon = PokemonPredictor()
    team.append(pokemon)


team = set(team)

print(f"after 10000 iterations, we got {len(team)} unique pokemons in the team")



# PredData = data.copy()

# PredData['Predicted Name'] = PredData.apply(lambda row: PokemonPredictor(
#     health_point=row['HP'],
#     attack=row['Attack'],
#     defense=row['Defense'],
#     sp_atk=row['Sp. Atk'],
#     sp_def=row['Sp. Def'],
#     speed=row['Speed']
# ), axis=1)








    

In [15]:
PredData['Right Prediction'] = PredData['Name'] == PredData['Predicted Name']

print(PredData['Right Prediction'].value_counts())

PredData[PredData['Right Prediction'] == False][['Name', 'Predicted Name', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']].head(30)

Right Prediction
True     778
False     22
Name: count, dtype: int64


,Name,Predicted Name,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
5,Charmeleon,Quilava,58,64,58,80,65,80
6,Charizard,Typhlosion,78,84,78,109,85,100
165,Mew,Celebi,100,100,100,100,100,100
169,Cyndaquil,Charmander,39,52,43,60,50,65
291,Cascoon,Silcoon,50,35,55,25,25,15
427,Jirachi,Celebi,100,100,100,100,100,100
533,RotomWash Rotom,RotomHeat Rotom,50,65,107,105,107,86
534,RotomFrost Rotom,RotomHeat Rotom,50,65,107,105,107,86
535,RotomFan Rotom,RotomHeat Rotom,50,65,107,105,107,86
536,RotomMow Rotom,RotomHeat Rotom,50,65,107,105,107,86
